# Bayesian KAN — Standard Toy Regression Benchmark

Evaluates BayesianKAN (B-coef) against BayesianNN (Bayes-by-Backprop) on the
standard 1D regression problems used in the BNN benchmark literature
(Hernandez-Lobato & Adams 2015; Blundell et al. 2015; Gal & Ghahramani 2016)
and turbulence-relevant toy cases.

**Plot convention** (Hernandez-Lobato & Adams 2015, Figure 1):  
Training data as scatter points · Predictive mean as solid line ·
±2σ credible interval as shaded region · True function as dashed line.

**Metrics** (Hernandez-Lobato & Adams 2015, Table 1):  
RMSE and test NLL (Gaussian negative log-likelihood),
averaged over `N_SEEDS` independent random train/test splits.

| Case | Name | Uncertainty type | Community reference |
|---|---|---|---|
| H-L 1D | Cubic y=x³ | Epistemic (extrapolation) | Hernandez-Lobato 2015 |
| Case 3 | Heteroscedastic | Aleatoric | Gal 2016 |
| Case 4 | Sparse data | Epistemic (extrapolation) | Blundell 2015 |
| Case 5 | Cubic sparse/dense | Mixed | Hernandez-Lobato 2015 |
| Case 6 | Sinusoidal gap | Epistemic (gap) | Blundell 2015 |
| Case 7 | Quadratic center noise | Aleatoric | — |


## 1. Setup

In [ ]:
import sys, os
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from bkan.models import BayesianKAN, BayesianNN
from bkan.training.trainer import BNNTrainer, TrainingConfig
from bkan.data.toy_problems import (
    generate_case3_heteroscedastic,
    generate_case4_sparse,
    generate_case5_cubic,
    generate_case6_sinusoidal,
    generate_case7_quadratic_center_noise,
)

DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
N_SEEDS  = 5   # number of random splits; increase to 20 for final paper numbers
N_MC     = 100 # MC samples at test time

print(f'Device: {DEVICE} | Seeds: {N_SEEDS}')

## 2. Helper functions

In [ ]:
def get_configs():
    """Return training configs for BKAN and BNN."""
    common = dict(epochs=3000, batch_size=256, learning_rate=1e-3,
                  n_mc_samples=1, early_stopping_patience=300,
                  scheduler='cosine', verbose=False)
    cfg_bkan = TrainingConfig(kl_annealing_epochs=300,  **common)
    cfg_bnn  = TrainingConfig(kl_annealing_epochs=1000, **common)
    return cfg_bkan, cfg_bnn


def train_bkan(x_tr, y_tr, grid_range, seed):
    torch.manual_seed(seed)
    model = BayesianKAN(input_dim=1, hidden_dims=[8, 8], output_dim=1,
                        num=5, k=3, prior_std=1.0, learn_noise=True,
                        grid_range=grid_range, device=DEVICE)
    cfg, _ = get_configs()
    BNNTrainer(model, cfg, DEVICE).train(x_tr, y_tr)
    return model


def train_bnn(x_tr, y_tr, seed):
    torch.manual_seed(seed)
    model = BayesianNN(input_dim=1, hidden_dims=[50, 50], output_dim=1,
                       prior_std=1.0, learn_noise=True, activation='tanh')
    _, cfg = get_configs()
    BNNTrainer(model, cfg, DEVICE).train(x_tr, y_tr)
    return model


def compute_metrics(mean_np, std_np, y_true_np):
    """RMSE and test NLL (Hernandez-Lobato & Adams 2015 metrics)."""
    rmse = np.sqrt(np.mean((mean_np - y_true_np) ** 2))
    # Gaussian NLL
    nll  = 0.5 * np.mean(np.log(2 * np.pi * std_np**2 + 1e-8)
                          + (mean_np - y_true_np)**2 / (std_np**2 + 1e-8))
    lower = mean_np - 1.96 * std_np
    upper = mean_np + 1.96 * std_np
    cov95 = np.mean((y_true_np >= lower) & (y_true_np <= upper))
    return rmse, nll, cov95


def predict_model(model, x_te):
    x_dev = x_te.to(DEVICE)
    mean, std, _ = model.predict(x_dev, n_samples=N_MC)
    _, ep_std, al_std = model.predict_decomposed(x_dev, n_samples=N_MC)
    return (mean.cpu().numpy().flatten(),
            std.cpu().numpy().flatten(),
            ep_std.cpu().numpy().flatten(),
            al_std.cpu().numpy().flatten())


# Standard BNN uncertainty plot (Hernandez-Lobato 2015 style)
def plot_prediction(ax, x_np, y_true, x_tr_np, y_tr_np,
                    mean, std, label, color, y_lim=None):
    ax.fill_between(x_np, mean - 2*std, mean + 2*std,
                    alpha=0.25, color=color, label=r'$\pm 2\sigma$')
    ax.plot(x_np, mean,   color=color, lw=2.0, label='Mean')
    ax.plot(x_np, y_true, 'k--', lw=1.2, alpha=0.7, label='True')
    ax.scatter(x_tr_np, y_tr_np, s=5, alpha=0.3, color='k', zorder=4,
               label='Data')
    if y_lim is not None:
        ax.set_ylim(*y_lim)
    ax.set_title(label, fontsize=10)
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.legend(fontsize=7, ncol=2, loc='upper left')
    ax.grid(True, alpha=0.25)

print('Helpers defined.')

## 3. Hernandez-Lobato 1D benchmark: y = x³

Exact setup from Hernandez-Lobato & Adams (2015), Section 5.3:  
20 training points uniform in [−4, 4], ε ~ N(0, 9), test on [−6, 6].

In [ ]:
HL_RESULTS = {'bkan': [], 'bnn': []}

for seed in range(N_SEEDS):
    rng = np.random.RandomState(seed)
    x_tr_np = rng.uniform(-4, 4, size=20)
    y_tr_np = x_tr_np**3 + rng.normal(0, 3, size=20)
    x_te_np = np.linspace(-6, 6, 300)
    y_true   = x_te_np**3

    x_tr = torch.FloatTensor(x_tr_np.reshape(-1, 1))
    y_tr = torch.FloatTensor(y_tr_np.reshape(-1, 1))
    x_te = torch.FloatTensor(x_te_np.reshape(-1, 1))

    for tag, train_fn in [('bkan', lambda s: train_bkan(x_tr, y_tr, [-6.5, 6.5], s)),
                          ('bnn',  lambda s: train_bnn(x_tr, y_tr, s))]:
        model = train_fn(seed)
        mean_np, std_np, _, _ = predict_model(model, x_te)
        rmse, nll, cov = compute_metrics(mean_np, std_np, y_true)
        HL_RESULTS[tag].append({'rmse': rmse, 'nll': nll, 'cov': cov,
                                 'mean': mean_np, 'std': std_np})
        print(f'Seed {seed} {tag:5s}: RMSE={rmse:.3f} NLL={nll:.3f} Cov={cov:.3f}')

print('\nHernandez-Lobato 1D done.')

In [ ]:
# Figure: last seed prediction (representative)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
x_te_np = np.linspace(-6, 6, 300)
y_true   = x_te_np**3
rng      = np.random.RandomState(N_SEEDS - 1)
x_tr_np  = rng.uniform(-4, 4, 20)
y_tr_np  = x_tr_np**3 + rng.normal(0, 3, 20)

ylim = (-280, 280)
for ax, tag, color, label in [
    (axes[0], 'bkan', 'C0', 'BayesianKAN  [1,8,8,1]'),
    (axes[1], 'bnn',  'C1', 'BayesianNN   [1,50,50,1]'),
]:
    r = HL_RESULTS[tag][-1]
    plot_prediction(ax, x_te_np, y_true, x_tr_np, y_tr_np,
                    r['mean'], r['std'], label, color, y_lim=ylim)

fig.suptitle('Hernandez-Lobato 1D: y = x³  (training in [−4, 4])', fontsize=12)
plt.tight_layout()
plt.savefig('hl1d_predictions.pdf', bbox_inches='tight')
plt.show()

for tag in ('bkan', 'bnn'):
    rmse_arr = np.array([r['rmse'] for r in HL_RESULTS[tag]])
    nll_arr  = np.array([r['nll']  for r in HL_RESULTS[tag]])
    cov_arr  = np.array([r['cov']  for r in HL_RESULTS[tag]])
    print(f'{tag:5s}  RMSE={rmse_arr.mean():.3f}±{rmse_arr.std():.3f}'
          f'  NLL={nll_arr.mean():.3f}±{nll_arr.std():.3f}'
          f'  Cov95={cov_arr.mean():.3f}±{cov_arr.std():.3f}')

## 4. Tyler toy cases 3, 4, 5, 6, 7

These cover the same uncertainty types as the BNN benchmark literature.

In [ ]:
CASES = {
    'case3': {
        'name': 'Case 3 — Heteroscedastic (aleatoric)',
        'gen':  lambda s: generate_case3_heteroscedastic(seed=s),
        'grid': [-0.1, 3.6],
        'y_lim': (-0.5, 3.5),
        'true_fn': lambda x: 0.3 * x**2,
    },
    'case4': {
        'name': 'Case 4 — Sparse data (epistemic extrapolation)',
        'gen':  lambda s: generate_case4_sparse(seed=s),
        'grid': [-0.1, 3.6],
        'y_lim': (-0.5, 5.5),
        'true_fn': lambda x: x * (1 + 0.2 * x**2),
    },
    'case5': {
        'name': 'Case 5 — Cubic sparse/dense (mixed)',
        'gen':  lambda s: generate_case5_cubic(seed=s),
        'grid': [-3.1, 3.1],
        'y_lim': (-4.5, 4.5),
        'true_fn': lambda x: x**3,
    },
    'case6': {
        'name': 'Case 6 — Sinusoidal gap (epistemic gap)',
        'gen':  lambda s: generate_case6_sinusoidal(seed=s),
        'grid': [-0.6, 1.1],
        'y_lim': (-2.0, 3.0),
        'true_fn': lambda x: x + 0.3*np.sin(2*np.pi*x) + 0.3*np.sin(4*np.pi*x),
    },
    'case7': {
        'name': 'Case 7 — Quadratic centre noise (aleatoric)',
        'gen':  lambda s: generate_case7_quadratic_center_noise(seed=s),
        'grid': [-3.6, 3.6],
        'y_lim': (-2.0, 12.0),
        'true_fn': lambda x: x**2,
    },
}

ALL_RESULTS = {k: {'bkan': [], 'bnn': []} for k in CASES}

for case_key, case in CASES.items():
    print('\n' + '=' * 60)
    print(f'Running: {case["name"]}')
    print('=' * 60)

    for seed in range(N_SEEDS):
        x_tr, y_tr, x_te, _ = case['gen'](seed)

        for tag, train_fn in [
            ('bkan', lambda s: train_bkan(x_tr, y_tr, case['grid'], s)),
            ('bnn',  lambda s: train_bnn(x_tr, y_tr, s)),
        ]:
            model = train_fn(seed)
            mean_np, std_np, ep_np, al_np = predict_model(model, x_te)
            x_te_np = x_te.numpy().flatten()
            y_true  = case['true_fn'](x_te_np)
            rmse, nll, cov = compute_metrics(mean_np, std_np, y_true)
            ALL_RESULTS[case_key][tag].append({
                'rmse': rmse, 'nll': nll, 'cov': cov,
                'mean': mean_np, 'std': std_np,
                'ep': ep_np, 'al': al_np,
                'x_te': x_te_np, 'y_true': y_true,
                'x_tr': x_tr.numpy().flatten(),
                'y_tr': y_tr.numpy().flatten(),
            })
            print(f'  Seed {seed} {tag:5s}: RMSE={rmse:.3f} NLL={nll:.3f} Cov95={cov:.3f}')

print('\nAll cases done.')

In [ ]:
# ── Prediction figures (one row per case, BKAN left / BNN right) ─────────────
# Style: Hernandez-Lobato & Adams 2015, Figure 1

fig, axes = plt.subplots(len(CASES), 2, figsize=(12, 4 * len(CASES)))

for row, (case_key, case) in enumerate(CASES.items()):
    seed_idx = -1   # representative seed (last)
    for col, (tag, color) in enumerate([('bkan', 'C0'), ('bnn', 'C1')]):
        r     = ALL_RESULTS[case_key][tag][seed_idx]
        label = ('BayesianKAN [1,8,8,1]' if tag == 'bkan'
                 else 'BayesianNN  [1,50,50,1]')
        plot_prediction(
            axes[row, col],
            r['x_te'], r['y_true'],
            r['x_tr'], r['y_tr'],
            r['mean'], r['std'],
            label=f'{case["name"]}\n{label}',
            color=color,
            y_lim=case['y_lim'],
        )

plt.tight_layout()
plt.savefig('toy_cases_all.pdf', bbox_inches='tight')
plt.show()
print('Figure saved: toy_cases_all.pdf')

In [ ]:
# ── Summary table (Hernandez-Lobato & Adams 2015 format) ─────────────────────
# Columns: RMSE ± std, NLL ± std, Coverage 95% ± std

def summary_row(results_list):
    rmse = np.array([r['rmse'] for r in results_list])
    nll  = np.array([r['nll']  for r in results_list])
    cov  = np.array([r['cov']  for r in results_list])
    return (f'{rmse.mean():.3f}±{rmse.std():.3f}',
            f'{nll.mean():.3f}±{nll.std():.3f}',
            f'{cov.mean():.3f}±{cov.std():.3f}')

case_names = {
    'hl1d':  'y=x³ (H-L 1D)',
    'case3': 'Heteroscedastic',
    'case4': 'Sparse',
    'case5': 'Cubic sparse/dense',
    'case6': 'Sinusoidal gap',
    'case7': 'Quadratic centre noise',
}

all_cases = {'hl1d': HL_RESULTS, **ALL_RESULTS}

header = (f"{'Dataset':>25}  {'Model':>8}  "
          f"{'RMSE':>14}  {'NLL':>14}  {'Cov95':>14}")
print(header)
print('-' * len(header))

for ck, cname in case_names.items():
    res = all_cases[ck] if ck == 'hl1d' else ALL_RESULTS[ck]
    for tag in ('bkan', 'bnn'):
        rmse_s, nll_s, cov_s = summary_row(res[tag])
        print(f'{cname:>25}  {tag:>8}  {rmse_s:>14}  {nll_s:>14}  {cov_s:>14}')
    print()